<a href="https://colab.research.google.com/github/cs-nick06/DecisionTreeProjectPython/blob/main/input_validation_usS15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Task 15
# Login backend 15.1 and 15.2

# input and basic validation

import state


class LoginForm:
    # holds the cleaned email and password from the login form
    def __init__(self, email, password):
        self.email = email
        self.password = password


def is_blank(text):
    if text is None:
        return True
    if str(text).strip() == "":
        return True
    return False


def looks_like_email(email):
    email = str(email)
    if "@" not in email:
        return False
    if "." not in email:
        return False

    parts = email.split("@")
    if len(parts) != 2:
        return False

    before_at = parts[0].strip()
    after_at = parts[1].strip()

    if before_at == "" or after_at == "":
        return False

    return True


def password_is_ok(password):
    if is_blank(password):
        return False
    if len(password) < 6:
        return False
    return True


def check_login_input(form_data):
    # read login form and do simple checks

    raw_email = form_data.get("email", "")
    raw_password = form_data.get("password", "")

    if is_blank(raw_email):
        return False, None

    if is_blank(raw_password):
        return False, None

    if not looks_like_email(raw_email):
        return False, None

    if not password_is_ok(raw_password):
        return False, None

    cleaned_email = raw_email.strip()
    cleaned_password = raw_password.strip()

    login_form = LoginForm(cleaned_email, cleaned_password)

    return True, login_form



def read_users_from_file(file_path):
    # read user records from a text file in a "email,password" style
    users = []

    try:
        with open(file_path, "r") as f:
            for line in f:
                line = line.strip()
                if line == "":
                    continue

                parts = line.split(",")
                if len(parts) < 2:
                    continue

                email_in_file = parts[0].strip()
                password_in_file = parts[1].strip()

                users.append({
                    "email": email_in_file,
                    "password": password_in_file
                })
    except FileNotFoundError:
        return []

    return users


def find_user_in_list(users, email, password):
    for user in users:
        if user["email"] == email and user["password"] == password:
            return True
    return False


def try_login_with_document(login_form, file_path):
    # use cleaned login data and check it against the user file

    users = read_users_from_file(file_path)

    if not users:
        CURRENT_SESSION["logged_in"] = False
        CURRENT_SESSION["email"] = None
        return False

    email = login_form.email
    password = login_form.password

    match = find_user_in_list(users, email, password)

    if match:
        CURRENT_SESSION["logged_in"] = True
        CURRENT_SESSION["email"] = email
        return True
    else:
        CURRENT_SESSION["logged_in"] = False
        CURRENT_SESSION["email"] = None
        return False